# 📈 04. Predictive Job Market Forecasting

This notebook demonstrates the development, training, validation, and comparative plotting of two machine learning models used in the **FutureSkills-AI** dashboard:
1. **Linear Regression** (Baseline model showing long-term trends)
2. **Random Forest Regressor** (Ensemble tree model for capturing non-linear relationships)

### Objectives:
- Load job posting metrics directly from MongoDB.
- Aggregate postings chronologically by year and month.
- Perform a chronological train-test split (testing on the final 6 months).
- Compute validation metrics: Root Mean Squared Error (RMSE) and Mean Absolute Percentage Error (MAPE).
- Forecast job trends 14 months into the future.
- Visualize actuals vs. model forecasts side-by-side with 95% Confidence Interval envelopes.

### 1. Import Dependencies

In [ ]:
import os
import numpy as np
import pandas as pd
from pymongo import MongoClient
from dotenv import load_dotenv
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for modern charts
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.family"] = "sans-serif"

### 2. Connect to MongoDB and Load Data

In [ ]:
# Load environment variables
load_dotenv(dotenv_path="../.env")

MONGO_URI = os.getenv("MONGO_URI")
if not MONGO_URI:
    # Fallback to localhost if no ENV file is found
    MONGO_URI = "mongodb://localhost:27017"

client = MongoClient(MONGO_URI)
# Explicitly connect to the unified database
db = client["futureskills_ai"]
jobs_col = db["jobs"]

print(f"Connected to MongoDB Atlas: {jobs_col.count_documents({})} jobs found in database.")

### 3. Chronological Aggregation
We aggregate job postings monthly to create a sequential time series.

In [ ]:
pipeline = [
    {
        "$group": {
            "_id": {
                "year": "$job_posting_year",
                "month": "$job_posting_month"
            },
            "jobs": {"$sum": 1}
        }
    }
]

results = list(jobs_col.aggregate(pipeline))
# Sort chronologically by year and month
results_sorted = sorted(results, key=lambda x: (x["_id"]["year"], x["_id"]["month"]))

data = []
for idx, r in enumerate(results_sorted):
    data.append({
        "index": idx,
        "date": f"{r['_id']['year']}-{r['_id']['month']:02d}",
        "value": r["jobs"]
    })

df = pd.DataFrame(data)
print("Sample time series data:")
print(df.head())

### 4. Chronological Train-Test Split
Since time-series predictions rely on sequence, we cannot perform random splits. We train on the history and evaluate on the final 6 months.

In [ ]:
X = df["index"].values.reshape(-1, 1)
y = df["value"].values

split_idx = len(df) - 6
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Training set size: {len(X_train)} months")
print(f"Testing set size (Evaluation Window): {len(X_test)} months")

### 5. Model Validation & Evaluation

#### Model 1: Linear Regression

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_test_lr = lr_model.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_test_lr))
mape_lr = mean_absolute_percentage_error(y_test, y_pred_test_lr) * 100

print(f"Linear Regression -")
print(f"  RMSE: {rmse_lr:.2f} jobs")
print(f"  MAPE: {mape_lr:.2f}%")

#### Model 2: Random Forest Regressor

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_test_rf = rf_model.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_test_rf))
mape_rf = mean_absolute_percentage_error(y_test, y_pred_test_rf) * 100

print(f"Random Forest Regressor -")
print(f"  RMSE: {rmse_rf:.2f} jobs")
print(f"  MAPE: {mape_rf:.2f}%")

### 6. Validation Summary

In [ ]:
summary_df = pd.DataFrame({
    "Model": ["Linear Regression (Baseline)", "Random Forest Regressor (Ensemble)"],
    "Validation RMSE (Jobs)": [round(rmse_lr, 2), round(rmse_rf, 2)],
    "Validation MAPE (%)": [f"{mape_lr:.2f}%", f"{mape_rf:.2f}%"]
})
summary_df

### 7. 14-Month Forecasting
We retrain both models on the complete dataset to predict future job index steps.

In [ ]:
# Retrain LR on full set
lr_full = LinearRegression()
lr_full.fit(X, y)
y_fit_lr = lr_full.predict(X)
residuals_lr = y - y_fit_lr
std_err_lr = np.std(residuals_lr)

# Retrain RF on full set
rf_full = RandomForestRegressor(n_estimators=100, random_state=42)
rf_full.fit(X, y)
y_fit_rf = rf_full.predict(X)
residuals_rf = y - y_fit_rf
std_err_rf = np.std(residuals_rf)

# Generate future steps
horizon = 14
last_idx = df["index"].iloc[-1]
future_X = np.array([last_idx + i for i in range(1, horizon + 1)]).reshape(-1, 1)

future_preds_lr = lr_full.predict(future_X)
future_preds_rf = rf_full.predict(future_X)

# Print projections
forecast_df = pd.DataFrame({
    "Future Month Step": np.arange(1, horizon + 1),
    "LR Forecast": np.round(np.clip(future_preds_lr, 0, None), 2),
    "RF Forecast": np.round(np.clip(future_preds_rf, 0, None), 2)
})
forecast_df

### 8. Visualization
Plot actuals along with forecasting curves and confidence limits.

In [ ]:
# Timeline alignment
hist_dates = df["date"].tolist()
last_year, last_month = map(int, df["date"].iloc[-1].split("-"))

future_dates = []
cy, cm = last_year, last_month
for _ in range(horizon):
    cm += 1
    if cm > 12:
        cm = 1
        cy += 1
    future_dates.append(f"{cy}-{cm:02d}")

all_dates = hist_dates + future_dates

# Plotting
plt.figure(figsize=(14, 7))

# Historical actuals
plt.plot(hist_dates, y, marker='o', label='Historical Job Postings', color='#2563eb', linewidth=2.5)

# Forecast curves starting from the last actual point
plt.plot([hist_dates[-1]] + future_dates, [y[-1]] + list(future_preds_lr), 
         linestyle='--', color='#10b981', linewidth=2.5, label='Linear Regression Forecast')

plt.plot([hist_dates[-1]] + future_dates, [y[-1]] + list(future_preds_rf), 
         linestyle='--', color='#8b5cf6', linewidth=2.5, label='Random Forest Forecast')

# 95% Confidence envelope for Linear Regression
lr_upper = np.array(list(future_preds_lr)) + 1.96 * std_err_lr
lr_lower = np.clip(np.array(list(future_preds_lr)) - 1.96 * std_err_lr, 0, None)

plt.fill_between(future_dates, lr_lower, lr_upper, color='#94a3b8', alpha=0.15, label='LR 95% Confidence Interval')

# Formats
plt.title("Job Postings Trend Analysis & ML Model Forecast Comparison", fontsize=16, fontweight='bold', pad=15)
plt.xlabel("Date (Year-Month)", fontsize=12, labelpad=10)
plt.ylabel("Number of Job Postings", fontsize=12, labelpad=10)
plt.xticks(rotation=45, ha='right')
plt.legend(frameon=True, facecolor='white', edgecolor='none', shadow=True, fontsize=11)
plt.tight_layout()

# Display plot
plt.show()